In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import jensenshannon

print("All imports successful")
print(f"scanpy version: {sc.__version__}")
print(f"anndata version: {ad.__version__}")

All imports successful
scanpy version: 1.11.5
anndata version: 0.11.4


/var/folders/1l/n6jztv_d3vdc37ddyfgc6x9r0000gp/T/ipykernel_75396/2576941680.py:10: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f"scanpy version: {sc.__version__}")


In [2]:
# Load the tonsil AnnData object
adata = sc.read_h5ad('../data/adata_nn_demo_tonsil.h5ad')

# First look at the object
print(adata)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '../data/adata_nn_demo_tonsil.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [3]:
# Load the tonsil AnnData object
adata = sc.read_h5ad('/Users/home/nsi-project/data/adata_nn_demo_tonsil.h5ad')

# First look at the object
print(adata)


AnnData object with n_obs × n_vars = 46789 × 59
    obs: 'DAPI', 'x', 'y', 'area', 'region_num', 'unique_region', 'condition', 'leiden_1', 'leiden_1_subcluster', 'cell_type_coarse', 'cell_type_coarse_subcluster', 'cell_type_coarse_f', 'cell_type_coarse_f_subcluster', 'cell_type_coarse_f_f', 'cell_type', 'CN_k20_n6', 'CN_k20_n6_annot'
    uns: 'CN_k20_n6_colors', 'Centroid_k20_n6', 'cell_type_coarse_f_colors', 'cell_type_colors', 'dendrogram_cell_type_coarse_f_subcluster', 'leiden', 'leiden_1_colors', 'neighbors', 'ppa_result_100', 'ppa_result_150', 'ppa_result_200', 'ppa_result_250', 'ppa_result_50', 'triDist', 'triDist_keyname', 'umap', 'unique_region_colors'
    obsm: 'X_pca', 'X_umap'
    layers: 'scaled'
    obsp: 'connectivities', 'distances'


In [4]:
# How many cells per CN?
print("CN distribution:")
print(adata.obs['CN_k20_n6_annot'].value_counts())

print("\nCell type distribution:")
print(adata.obs['cell_type'].value_counts())

print("\nConditions in dataset:")
print(adata.obs['condition'].value_counts())


CN distribution:
CN_k20_n6_annot
Immune Priming Zone            11485
Marginal Zone B-DC-Enriched    10259
Marginal Zone                  10226
Parafollicular T cell Zone      8148
Epithelium                      3898
Germinal Center                 2773
Name: count, dtype: int64

Cell type distribution:
cell_type
B cell             15733
CD4+ T cell         7376
DC                  3642
Vessel              3332
Epithelial cell     3322
CD8+ T cell         2851
GCB                 2567
Plasma cell         2486
M2 Macrophage       1634
Treg                1279
cDC1                1038
M1 Macrophage        969
Mast cell            301
NK cell              141
Neutrophil           118
Name: count, dtype: int64

Conditions in dataset:
condition
tonsil         26507
tonsillitis    20282
Name: count, dtype: int64


In [5]:
import pandas as pd

# The centroids are stored in adata.uns
centroids = pd.DataFrame(
    adata.uns['Centroid_k20_n6'],
    index=adata.obs['CN_k20_n6_annot'].cat.categories
)

print("CN Centroid table (rows = CNs, columns = cell types):")
print(centroids.round(3))
print(f"\nShape: {centroids.shape}")

ValueError: Data must be 1-dimensional, got ndarray of shape (6, 16) instead

In [6]:
# Inspect the raw centroid structure
print(type(adata.uns['Centroid_k20_n6']))
print()
print(adata.uns['Centroid_k20_n6'])

<class 'dict'>

{'20': array([[6.74846626e-02, 1.97296382e+00, 1.96027078e+00, 2.26354982e+00,
        3.57732177e-01, 4.67526973e-02, 1.67336577e+00, 3.50035964e+00,
        2.19509202e+00, 1.06118045e+00, 1.10962556e+00, 2.58354136e+00,
        6.59107256e-01, 3.68394330e-01, 1.19272266e-01, 6.13073831e-02],
       [2.77964152e-03, 1.84510687e-02, 1.67549123e+01, 6.68072462e-01,
        7.55774945e-02, 7.67037286e-01, 4.51691747e-01, 3.83111282e-01,
        5.36279114e-01, 4.82124030e-02, 2.08473114e-02, 1.05722228e-01,
        1.51154989e-01, 1.15019649e-03, 1.40899070e-02, 9.10572223e-04],
       [5.16449410e-02, 4.29546865e-02, 1.75319677e+00, 2.84792055e-01,
        1.41976412e+01, 7.82122905e-03, 9.77032899e-02, 2.92240844e-01,
        7.73805090e-01, 2.26443203e-01, 2.22222222e-02, 1.88404718e+00,
        2.87026691e-01, 4.04717567e-02, 3.20297952e-02, 5.95903166e-03],
       [3.92317602e-03, 9.64808526e-01, 1.56786509e+00, 1.00009369e+01,
        7.07342780e-02, 3.97001991e-02

In [7]:
# Check the CN category order
print("CN categories:")
print(adata.obs['CN_k20_n6_annot'].cat.categories.tolist())

print("\nCell type categories:")
print(adata.obs['cell_type'].cat.categories.tolist())

# Check what CN_k20_n6 (numeric) looks like
print("\nNumeric CN labels (first 10):")
print(adata.obs['CN_k20_n6'].head(10))

CN categories:
['Epithelium', 'Germinal Center', 'Immune Priming Zone', 'Marginal Zone', 'Marginal Zone B-DC-Enriched', 'Parafollicular T cell Zone']

Cell type categories:
['B cell', 'CD4+ T cell', 'CD8+ T cell', 'DC', 'Epithelial cell', 'GCB', 'M1 Macrophage', 'M2 Macrophage', 'Mast cell', 'NK cell', 'Neutrophil', 'Plasma cell', 'Treg', 'Vessel', 'cDC1']

Numeric CN labels (first 10):
0     5
1     1
2     5
3     4
4     1
5     5
6     1
8     2
10    1
12    1
Name: CN_k20_n6, dtype: int32


In [8]:
# Compute CN composition from scratch
# For each CN, count what fraction of cells belong to each cell type

# Cross-tabulate CN assignments against cell types
cn_celltype = pd.crosstab(
    adata.obs['CN_k20_n6_annot'],
    adata.obs['cell_type'],
    normalize='index'  # normalize by row so each CN sums to 1
)

print("CN composition matrix (rows=CNs, columns=cell types):")
print(cn_celltype.round(3))
print(f"\nShape: {cn_celltype.shape}")
print(f"\nDo rows sum to 1? {cn_celltype.sum(axis=1).round(3).tolist()}")

CN composition matrix (rows=CNs, columns=cell types):
cell_type                    B cell  CD4+ T cell  CD8+ T cell     DC  \
CN_k20_n6_annot                                                        
Epithelium                    0.069        0.011        0.004  0.031   
Germinal Center               0.215        0.004        0.010  0.022   
Immune Priming Zone           0.086        0.097        0.080  0.110   
Marginal Zone                 0.858        0.032        0.021  0.023   
Marginal Zone B-DC-Enriched   0.437        0.156        0.074  0.137   
Parafollicular T cell Zone    0.076        0.525        0.113  0.069   

cell_type                    Epithelial cell    GCB  M1 Macrophage  \
CN_k20_n6_annot                                                      
Epithelium                             0.766  0.001          0.001   
Germinal Center                        0.000  0.741          0.000   
Immune Priming Zone                    0.010  0.002          0.055   
Marginal Zone      